# Lithuanian LRT summarization model

This notebook fine-tunes a small instruction model for Lithuanian summarization. The examples come from LRT articles: the article body is the input, and the existing LRT article summary is the target.

Using real summaries is much cleaner than generating labels from Wikipedia text, because the target is written as an actual summary rather than copied from the first sentence of the input.


## 1. Libraries and setup

The random seeds keep the split and training run reproducible enough for a small course project.


In [2]:
import os
import json
import random
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType

random.seed(42)
torch.manual_seed(42)
os.makedirs("data", exist_ok=True)


## 2. Load the LRT dataset

The data collection notebook saves LRT articles with these important fields:

- `text`: full article body used as model input;
- `summary`: LRT-provided summary used as the target;
- `title` and `url`: useful for inspection and leakage-safe splitting.


In [4]:
dataset_path = "data/lrt_summarization_200.jsonl"

raw_df = pd.read_json(dataset_path, lines=True)

instruction = "Apibendrink pateiktą lietuvišką naujienų tekstą keliais aiškiais sakiniais."

df = raw_df.rename(
    columns={
        "text": "input",
        "summary": "output",
    }
).copy()

df["instruction"] = instruction

df = df[
    ["instruction", "input", "output", "title", "url", "summary_words", "text_words", "text_hash"]
]

df = df.dropna(subset=["input", "output", "title", "url"])
df = df.drop_duplicates(subset=["url", "text_hash"]).reset_index(drop=True)

df.head()


,instruction,input,output,title,url,summary_words,text_words,text_hash
0,Apibendrink pateiktą lietuvišką naujienų tekst...,Kalvarijos gimnazijos rūsyje laikinai prisigla...,Priedangoms modernizuoti savivaldybėse vyriaus...,"Savivaldybėse atnaujinama civilinė sauga, siūl...",https://www.lrt.lt/naujienos/lietuvoje/2/29190...,37,369,206a10dd130f9176e82ba67bddd78e0b388b79aef05a2b...
1,Apibendrink pateiktą lietuvišką naujienų tekst...,"Visuomenininkas K. Žukauskas, kuris domėjosi „...",Visuomenininkui Karoliui Žukauskui ėmus skelbt...,Žukauskas tiria Puchovičiaus vizitus statybose...,https://www.lrt.lt/naujienos/lietuvoje/2/29193...,53,537,464efbe5e6b0d45db1f5db796aca393cab4a1dcc34193d...
2,Apibendrink pateiktą lietuvišką naujienų tekst...,„Dienos temoje“ – Lietuvos valstiečių ir žalių...,„Nuo pat pradžių mes tikrai skeptiškai buvome ...,Veryga apie Palucką: šio politiko atžvilgiu sk...,https://www.lrt.lt/naujienos/lietuvoje/2/29194...,42,1310,5a9b397696ae49caa7ca964f749955f073985e5ec99722...
3,Apibendrink pateiktą lietuvišką naujienų tekst...,„Turbūt šitoje situacijoje aš tikrai sutinku s...,Lietuvos valstiečių ir žaliųjų sąjungos pirmin...,Veryga: teisėsaugos dėmesio sulaukęs Gintautas...,https://www.lrt.lt/naujienos/lietuvoje/2/29193...,29,174,d9e0039a4953f20894035522cb7aa143e080841c427e7d...
4,Apibendrink pateiktą lietuvišką naujienų tekst...,Baltarusijos valstybiniame televizijos kanale ...,"Aktyvus protestų ir incidentų dalyvis, keliose...",Kandrotas Baltarusijoje jau spėjo išpilti purv...,https://www.lrt.lt/naujienos/lietuvoje/2/29193...,31,566,fc2d95a74b1fb6e9d69fe00708bc584d73be3e3559744a...


## 3. Dataset checks

Before training, I check the size and basic text lengths. These numbers are useful when explaining the dataset during assessment.


In [ ]:
print("Dataset size:", len(df))
print("Average input words:", round(df["text_words"].mean(), 1))
print("Average summary words:", round(df["summary_words"].mean(), 1))
print("Unique URLs:", df["url"].nunique())
print("Unique titles:", df["title"].nunique())
print("Summaries copied at start of input:", sum(input_text.startswith(output) for input_text, output in zip(df["input"], df["output"])))


Dataset size: 149
Average input words: 739.7
Average summary words: 37.2
Unique URLs: 149
Unique titles: 149
Summaries copied at start of input: 0


In [6]:
sample = df.sample(1, random_state=42).iloc[0]

print("TITLE:", sample["title"])
print("\nREFERENCE SUMMARY:\n", sample["output"])
print("\nINPUT PREVIEW:\n", sample["input"][:1500])


TITLE: Aistė Adomavičienė. Atsakymas Nerijui Mačiuliui arba Ar tikrai Lietuva rikiuotės gale?

REFERENCE SUMMARY:
 Sakoma, skurdas geriausiai išmoko skaičiuoti. Praėjusią savaitę daug žiniasklaidos dėmesio sulaukė paskelbta žinia apie naują metodiką, pagal kurią Lietuva – skurdžiausia Europos Sąjungos šalis. Pagal daugelį skurdo rodiklių Lietuva jau ne vienerius metus yra tarp prasčiausius rodiklius ES turinčių valstybių. Nors mums ir nesinori šios etiketės savo šaliai, tačiau naujausi Europos statistikos departamento (Eurostat) duomenys patvirtina, kad Lietuvoje skurdo rizikos lygis, kuris 2025 m. šoktelėjo iki 22,6 proc., yra didžiausias ES.

INPUT PREVIEW:
 Savo komentare šią temą palietė ir „Swedbank“ Lietuvoje vyriausiasis ekonomistas Nerijus Mačiulis tvirtinantis, kad skurdo rizikos lygis parodo „santykinį skurdą“ ir pajamų nelygybę, o ne tikrąjį nepriteklių. Ekonominė gerovė nepasiekia pažeidžiamų žmonių Iš dalies sutinkame su ekonomistu, rodiklis nėra tobulas, jam turi įtakos š

## 4. Save project metadata

The dataset is already collected, but this metadata file makes the project structure clearer.


In [7]:
metadata = {
    "dataset_name": "lrt_lithuanian_news_summarization",
    "language": "Lithuanian",
    "source": "LRT news articles collected by the data_collection notebook",
    "dataset_size": len(df),
    "task": "Summarize a Lithuanian news article using the article's existing LRT summary as the target",
    "structure": {
        "instruction": "Lithuanian instruction for the model",
        "input": "Article body text from LRT",
        "output": "Existing LRT article summary",
        "title": "Article title",
        "url": "Article URL",
    },
    "why_useful": "The summaries are human-written article summaries, so the targets are much cleaner than automatic extractive labels.",
    "notes": [
        "The train/test split is done by URL to avoid evaluating on the same article used during training.",
        "The dataset is intentionally small because it is for a compact LoRA fine-tuning demonstration.",
    ],
}

with open("data/metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Saved metadata for", len(df), "examples")


Saved metadata for 149 examples


## 5. Train/test split

The split is made by article URL. This keeps every article entirely in either train or test.


In [8]:
unique_urls = df["url"].drop_duplicates().tolist()
random.Random(42).shuffle(unique_urls)

test_url_count = max(1, round(len(unique_urls) * 0.16))
test_urls = set(unique_urls[:test_url_count])

train_df = df[~df["url"].isin(test_urls)].reset_index(drop=True)
test_df = df[df["url"].isin(test_urls)].reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

print(train_dataset)
print(test_dataset)
print("Overlapping URLs:", sorted(set(train_df["url"]) & set(test_df["url"])))


Dataset({
    features: ['instruction', 'input', 'output', 'title', 'url', 'summary_words', 'text_words', 'text_hash'],
    num_rows: 125
})
Dataset({
    features: ['instruction', 'input', 'output', 'title', 'url', 'summary_words', 'text_words', 'text_hash'],
    num_rows: 24
})
Overlapping URLs: []


## 6. Prompt format

The same chat-style prompt is used during training and inference.


In [9]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


def make_prompt(example):
    messages = [
        {
            "role": "system",
            "content": "Tu esi lietuviškų tekstų apibendrinimo asistentas. Atsakyk tik santrauka lietuvių kalba."
        },
        {
            "role": "user",
            "content": (
                f"{example['instruction']}\n\n"
                f"Tekstas:\n{example['input']}"
            )
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def add_prompt_columns(example):
    prompt = make_prompt(example)
    return {
        "prompt": prompt,
        "answer": example["output"],
        "text": prompt + example["output"],
    }


train_dataset = train_dataset.map(add_prompt_columns)
test_dataset = test_dataset.map(add_prompt_columns)


Map: 100%|██████████| 24/24 [00:00<00:00, 2697.30 examples/s]


## 7. Model and LoRA setup

I use the same base model and LoRA approach as before. Only the dataset changed.


In [10]:
if torch.cuda.is_available():
    dtype = torch.float16
else:
    dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=dtype,
)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 498.73it/s]


In [16]:
max_length = 1024


def tokenize(example):
    prompt_tokens = tokenizer(example["prompt"], add_special_tokens=False)
    full_text = example["prompt"] + example["answer"] + tokenizer.eos_token

    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=max_length,
        padding=False,
    )

    labels = tokens["input_ids"].copy()
    prompt_length = len(prompt_tokens["input_ids"])
    prompt_length = min(prompt_length, len(labels))

    for index in range(prompt_length):
        labels[index] = -100

    tokens["labels"] = labels
    return tokens


train_tokenized = train_dataset.map(tokenize, remove_columns=train_dataset.column_names)
test_tokenized = test_dataset.map(tokenize, remove_columns=test_dataset.column_names)


Map: 100%|██████████| 24/24 [00:00<00:00, 96.20 examples/s] 


In [17]:
loraconfig = LoraConfig(
    r=8, # Reduced from 16
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, loraconfig)
model.print_trainable_parameters()


c:\Users\doman\AppData\Local\Programs\Python\Python310\lib\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\Users\doman\AppData\Local\Programs\Python\Python310\lib\site-packages\peft\tuners\tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


## 8. Training

Training stays deliberately small because this is a course-sized LoRA demonstration.


In [19]:
training_args = TrainingArguments(
    output_dir="lt_summary_lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=1e-4,
    logging_steps=10,
    save_strategy="no",
    eval_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
)

trainer.train()


Step,Training Loss
10,10.363632
20,7.719552
30,11.075705
40,4.082055
50,7.003271
60,3.893293
70,5.617859
80,2.155568
90,2.538407


TrainOutput(global_step=96, training_loss=5.814481129248937, metrics={'train_runtime': 153.11, 'train_samples_per_second': 2.449, 'train_steps_per_second': 0.627, 'total_flos': 801043492142592.0, 'train_loss': 5.814481129248937, 'epoch': 3.0})

In [20]:
model.save_pretrained("lt_summary_lora_adapter")
tokenizer.save_pretrained("lt_summary_lora_adapter")


('lt_summary_lora_adapter\\tokenizer_config.json',
 'lt_summary_lora_adapter\\chat_template.jinja',
 'lt_summary_lora_adapter\\tokenizer.json')

## 9. Demonstration

This helper uses the same prompt style as training and returns only the generated summary.


In [21]:
def summarize_lithuanian(text, max_new_tokens=120):
    model.eval()

    messages = [
        {
            "role": "system",
            "content": "Tu esi lietuviškų tekstų apibendrinimo asistentas. Atsakyk tik santrauka lietuvių kalba."
        },
        {
            "role": "user",
            "content": (
                "Apibendrink pateiktą lietuvišką naujienų tekstą keliais aiškiais sakiniais.\n\n"
                f"Tekstas:\n{text}"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    generated = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return generated.strip().split("###")[0].strip()


In [22]:
example = test_dataset[0]

print("TITLE:", example["title"])
print("\nINPUT:")
print(example["input"][:1500])

print("\nREFERENCE SUMMARY:")
print(example["output"])

print("\nMODEL SUMMARY:")
print(summarize_lithuanian(example["input"]))


TITLE: Pareigūnai: Suomijos oro erdvę pažeidę dronai greičiausiai yra ukrainiečių

INPUT:
Dronai į Suomijos oro erdvę įskrido iš pietų ir skrido į šiaurės rytus, į Rusijos teritoriją, tačiau kur jie atsidūrė, nežinoma, nurodė pasienio apsaugos pareigūnai. Įtariami oro erdvės pažeidimai įvyko rytinėje Suomijos įlankos dalyje, netoli Suomijos 1 340 kilometrų ilgio sienos su Rusija. Rusijos gynybos ministerija sekmadienį pranešė, kad jos karinės oro erdvės gynybos daliniai per naktį maždaug 15 regionų perėmė 334 ukrainiečių dronus. Kai kurie jų buvo netoli Sankt Peterburgo, maždaug už 150 km nuo Suomijos Virolahčio savivaldybės, kurioje buvo pastebėtas vienas iš dronų. Suomijos nacionalinis tyrimų biuras sekmadienį įvykusius įtariamus dronų įsibrovimus tiria kaip „sunkų pavojaus visuomenės saugumui sukėlimą“, o Suomijos pasienio apsaugos pareigūnai yra atsakingi už įtariamų teritorijos pažeidimų tyrimą. Jie taip pat tiria keturių nuklydusių ukrainiečių dronų, kurie kovo pabaigoje ir balan